# ReasonForge — GRPO + LoRA in Google Colab
This notebook checks the runtime, installs the pinned stack, prepares leakage-safe GSM8K subsets, runs CPU-safe tests, trains a Qwen2.5-0.5B LoRA adapter, performs paired held-out evaluation, and launches the app. **Training and evaluation cells are GPU-expensive.**

In [ ]:
# 1. Runtime check — select Runtime > Change runtime type > T4 GPU first.
import platform
import subprocess
import torch
print('Python:', platform.python_version())
subprocess.run(['nvidia-smi'], check=False)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime before continuing to training.')

In [ ]:
# 2. Clone a published repository, or open an uploaded /content/ReasonForge directory.
from pathlib import Path
import os, subprocess
REPO_URL = 'https://github.com/jonathannerd/ReasonForge.git'
PROJECT_DIR = Path('/content/ReasonForge')
if REPO_URL and not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
if not PROJECT_DIR.exists():
    raise FileNotFoundError('Set REPO_URL or upload ReasonForge to /content/ReasonForge')
os.chdir(PROJECT_DIR)
print('Project:', Path.cwd())

In [ ]:
# 3. Install the pinned training stack. Restart the runtime if pip requests it.
%pip install -q -r requirements-colab.txt
%pip install -q -e . pytest==8.4.2 ruff==0.13.1
# Colab may preinstall torchvision/torchaudio wheels for a newer torch release.
# ReasonForge is text-only; removing those unused, incompatible wheels keeps Transformers importable.
%pip uninstall -q -y torchvision torchaudio

In [ ]:
# 4. Create a fast demonstration config without changing the checked-in config.
import yaml
from pathlib import Path
config = yaml.safe_load(Path('configs/training.yaml').read_text())
config['dataset'].update(train_size=32, validation_size=8, test_size=8)
config['training'].update(max_steps=3, save_steps=3, output_dir='outputs/reasonforge-demo-adapter')
Path('/content/reasonforge_demo.yaml').write_text(yaml.safe_dump(config, sort_keys=False))
print(Path('/content/reasonforge_demo.yaml').read_text())

In [ ]:
# 5. Prepare data and run the CPU-safe smoke suite before spending GPU time.
!python -m reasonforge.dataset --config /content/reasonforge_demo.yaml --output-dir outputs/demo-dataset
!pytest -q
import sys
sys.path.insert(0, str(PROJECT_DIR / 'src'))  # expose the editable source to this live kernel
from reasonforge.rewards import score_completion
sample = '{"method":"multiply","calculations":[{"expression":"12*5","result":"60"}],"final_answer":"60"}'
score_completion(sample, '60')

## GPU-expensive: choose one training run
The demo is a pipeline check, not a meaningful experiment. The default checked-in config uses 512 prompts and 100 steps. Start with the demo; use `--fallback` if a T4 runs out of memory.

In [ ]:
# 6A. FAST DEMONSTRATION — three GRPO steps.
!python -m reasonforge.train --config /content/reasonforge_demo.yaml --fallback

In [ ]:
# 6B. MORE MEANINGFUL EXPLORATORY RUN — skip if you ran only the demo.
# !python -m reasonforge.train --config configs/training.yaml
# Resume after interruption with: ... --resume

In [ ]:
# 7. Confirm that the LoRA adapter and metadata were saved.
from pathlib import Path
adapter = Path('outputs/reasonforge-demo-adapter')
assert (adapter / 'adapter_config.json').exists(), list(adapter.glob('*'))
print('Saved files:', [p.name for p in adapter.iterdir()])

## GPU-expensive: paired base-versus-adapter evaluation
The evaluation reloads the base and aligned policies sequentially and writes only measured artifacts. Point a copied evaluation config at the demo adapter.

In [ ]:
# 8–9. Evaluate and render the generated comparison plot.
evaluation = yaml.safe_load(Path('configs/evaluation.yaml').read_text())
evaluation['dataset'].update(train_size=32, validation_size=8, test_size=8)
evaluation['evaluation'].update(adapter_path='outputs/reasonforge-demo-adapter', output_dir='results/demo')
Path('/content/reasonforge_eval_demo.yaml').write_text(yaml.safe_dump(evaluation, sort_keys=False))
!python -m reasonforge.evaluate --config /content/reasonforge_eval_demo.yaml
from IPython.display import Image, display
display(Image(filename='results/demo/comparison.png'))

In [ ]:
# 10. Launch Gradio. Stop this cell to continue.
!python -m reasonforge.app --adapter-path outputs/reasonforge-demo-adapter --share

In [ ]:
# 11. Optional: save adapter and evaluation artifacts to Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# destination = '/content/drive/MyDrive/ReasonForge-artifacts'
# shutil.copytree('outputs/reasonforge-demo-adapter', destination + '/adapter', dirs_exist_ok=True)
# shutil.copytree('results/demo', destination + '/evaluation', dirs_exist_ok=True)